In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import lamindb as ln
import squidpy as sq

ln.track("mv6JfoOwmOUa")

In [ ]:
FIGURES_DIR = '/home/workspace/PCD_data/xenium/commercial/figures/qc_calibration'

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
def save_fig(fig, stem, description, ulabels=(), features=None):
    for ax in fig.axes:
        for coll in ax.collections:
            coll.set_rasterized(True)
    for ext in ['png', 'pdf']:
        path = os.path.join(FIGURES_DIR, f'{stem}.{ext}')
        fig.savefig(path, dpi=150, bbox_inches='tight')
        artifact = ln.Artifact(path, key=f'{stem}.{ext}', description=description).save()
        if ulabels:
            artifact.ulabels.add(*ulabels)
        if features:
            artifact.features.add_values(features)
    plt.close(fig)

In [ ]:
SAMPLE_NAMES = [
    'TIS05393',
    'TIS04788',
    'TIS05392',
    'TIS05394',
    'TIS04789',
    'TIS05397_008',
]

QC_COLUMNS = ['total_counts', 'n_genes_by_counts']

In [ ]:
samples = {}
for name in SAMPLE_NAMES:
    # Loads the raw per-sample AnnData from lamindb storage.
    adata = ln.Artifact.get(key=f'{name}.h5ad').load()
    # Adds obs columns total_counts and n_genes_by_counts (computed from .X, not stored).
    sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)
    samples[name] = adata

combined = sc.concat(samples, label='sample')

combined

In [ ]:
# One row per sample: counts histogram, genes histogram, and knee plot (rank vs counts).
# A sharp bend in the knee marks a natural cutoff between real cells and debris.
fig, axes = plt.subplots(len(samples), 3, figsize=(15, 4 * len(samples)), squeeze=False, gridspec_kw={'hspace': 0.4})
for row_axes, (name, adata) in zip(axes, samples.items()):
    counts = adata.obs['total_counts'].to_numpy()
    genes  = adata.obs['n_genes_by_counts'].to_numpy()

    row_axes[0].hist(np.log10(counts + 1), bins=60, color='steelblue')
    row_axes[0].set_xlabel('log10(counts + 1)')
    row_axes[0].set_title(f'{name} — counts  (median {np.median(counts):.0f})')

    row_axes[1].hist(np.log10(genes + 1), bins=60, color='steelblue')
    row_axes[1].set_xlabel('log10(genes + 1)')
    row_axes[1].set_title(f'{name} — genes  (median {np.median(genes):.0f})')

    row_axes[2].plot(np.arange(1, len(counts) + 1), np.sort(counts)[::-1], lw=1.5)
    row_axes[2].set_xscale('log')
    row_axes[2].set_yscale('log')
    row_axes[2].set_xlabel('cell rank')
    row_axes[2].set_ylabel('counts')
    row_axes[2].set_title(f'{name} — knee')

plt.show()

qc_label        = ln.ULabel(name='qc').save()
histogram_label = ln.ULabel(name='histogram').save()
save_fig(fig, 'qc_distributions', 'counts/genes histogram and count knee plot',
         ulabels=(qc_label, histogram_label))

In [ ]:
# One row per sample, one column per QC metric, plotted at tissue coordinates (color capped at p99).
# Low counts at tissue edges are normal; a diffuse low-count haze points to a run problem.

fig, axes = plt.subplots(
    len(samples), len(QC_COLUMNS),
    figsize=(5.5 * len(QC_COLUMNS), 5 * len(samples)),
    squeeze=False,
)

for i, (name, adata) in enumerate(samples.items()):
    for j, column in enumerate(QC_COLUMNS):
        ax = axes[i, j]
        sc.pl.spatial(
            adata, color=column, vmax='p99', spot_size=50,
            ax=ax, show=False, colorbar_loc=None,
            title=f'{name} — {column}\n(median {adata.obs[column].median():.0f})',
        )
        fig.colorbar(
            ax.collections[0],
            cax=ax.inset_axes([1.03, 0, 0.03, 1]),
        )

fig.tight_layout()
plt.show()

# spatial_label   = ln.ULabel(name='spatial').save()
# save_fig(fig, 'spatial_qc_maps', 'spatial QC maps, total counts and genes by counts per sample',
#          ulabels=(qc_label, spatial_label))

In [ ]:
nrows, ncols = 2, 3
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = axes.flatten()

for ax, (name, adata) in zip(axes, samples.items()):
    sc.pl.highest_expr_genes(adata, n_top=10, ax=ax, show=False)
    ax.set_title(name)

for ax in axes[len(samples):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
for name, adata in samples.items():
    X = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
    row_sums = X.sum(axis=1, keepdims=True)
    fracs = X / np.where(row_sums == 0, 1, row_sums)
    median_fracs = np.nanmedian(fracs, axis=0)
    top10 = list(adata.var_names[np.argsort(median_fracs)[-10:][::-1]])
    sc.pl.spatial(adata, color=top10, use_raw=False, img_key=None, spot_size=6, ncols=5,
                  vmax='p99', title=[f'{name} — {g}' for g in top10], show=True)

In [ ]:
MIN_COUNTS = 15
MIN_GENES  = 10

# Adds layers['counts']: a copy of the raw integer counts, since normalize_total overwrites .X.
combined.layers['counts'] = combined.X.copy()

sc.pp.filter_cells(combined, min_counts=MIN_COUNTS)
sc.pp.filter_cells(combined, min_genes=MIN_GENES)

ln.Param(name='min_counts', dtype=int).save()
ln.Param(name='min_genes',  dtype=int).save()
filtered_label = ln.ULabel(name='filtered').save()

artifact = ln.Artifact.from_anndata(combined, key='test_combined_filtered.h5ad',
                                    description='Combined Xenium object after QC cell filtering').save()
artifact.features.add_values({'min_counts': MIN_COUNTS, 'min_genes': MIN_GENES, 'harmony': False})
artifact.ulabels.add(qc_label, filtered_label)

ln.finish()